RAW to JPEG photo conversion workflow

Workflow design
- Workflow for local and HPC processing (see info below)
- Designed to ouput JPGs with following specifications (300 dpi, 8256 x 5504 px dimensions)

HPC processing 
- Full workflow description here:  
https://aimsgovau.sharepoint.com/sites/3Dmodelling/_layouts/15/Doc.aspx?sourcedoc={bf6b30ef-9fc4-4a12-8d8d-5e73567ec3dd}&action=edit&wd=target%28General.one%7Cfd467a86-1542-4f91-be13-26e1d02f5985%2FHPC%20jobs%7C28f11517-fe1e-46c6-99e4-f817ca7c81c1%2F%29&wdorigin=703
- Contact IT to set up HPC account/access
- Windows 10: Install Windows Terminal from Microsoft Store 
- Windows 11: Terminal inbuilt and WSL2 compatible (no need to install) 
- Time to run: <5 seconds per photo

Local processing 
- See file below to setup VS code and modules if not yet set up 
    -https://aimsgovau.sharepoint.com/:u:/s/3Dmodelling/Eb9jxHImCxNEib6tL3qzUmUBDJ6QtOejU1SV71lEIUg4vA?e=zMmPfz
- Install modules rawpy and imagio in environment in Andaconda Prompt before use (pip install rawpy imageio) 
- Script prompts user to provide input and output location of files
- Time to run: ~15-20 seconds per photo

RAW types
- Nikon D850: NEF, 8256 x 5504 px, 300 dpi
- Sony ILCE-6700: ARW, 6192 x 4128 px. 240 dpi 


RAW to JPEG conversion options (rawpy: raw.postprocess()) 
- Specifies how NEF/ARW is converted to JPEG

Description below from : https://letmaik.github.io/rawpy/api/rawpy.Params.html

classrawpy.Params(self, demosaic_algorithm=None, half_size=False, four_color_rgb=False, dcb_iterations=0, dcb_enhance=False, fbdd_noise_reduction=FBDDNoiseReductionMode.Off, noise_thr=None, median_filter_passes=0, use_camera_wb=False, use_auto_wb=False, user_wb=None, output_color=ColorSpace.sRGB, output_bps=8, user_flip=None, user_black=None, user_sat=None, no_auto_bright=False, auto_bright_thr=None, adjust_maximum_thr=0.75, bright=1.0, highlight_mode=HighlightMode.Clip, exp_shift=None, exp_preserve_highlights=0.0, no_auto_scale=False, gamma=None, chromatic_aberration=None, bad_pixels_path=None)¶
A class that handles postprocessing parameters.

If use_camera_wb and use_auto_wb are False and user_wb is None, then daylight white balance correction is used. If both use_camera_wb and use_auto_wb are True, then use_auto_wb has priority.

Parameters:
- demosaic_algorithm (rawpy.DemosaicAlgorithm) – default is AHD
- half_size (bool) – outputs image in half size by reducing each 2x2 block to one pixel instead of interpolating
- four_color_rgb (bool) – whether to use separate interpolations for two green channels
- dcb_iterations (int) – number of DCB correction passes, requires DCB demosaicing algorithm
- dcb_enhance (bool) – DCB interpolation with enhanced interpolated colors
- fbdd_noise_reduction (rawpy.FBDDNoiseReductionMode) – controls FBDD noise reduction before demosaicing
- noise_thr (float) – threshold for wavelet denoising (default disabled)
- median_filter_passes (int) – number of median filter passes after demosaicing to reduce color artifacts
- use_camera_wb (bool) – whether to use the as-shot white balance values
- use_auto_wb (bool) – whether to try automatically calculating the white balance
- user_wb (list) – list of length 4 with white balance multipliers for each color
- output_color (rawpy.ColorSpace) – output color space
- output_bps (int) – 8 or 16
- user_flip (int) – 0=none, 3=180, 5=90CCW, 6=90CW, default is to use image orientation from the RAW image if available
- user_black (int) – custom black level
- user_sat (int) – saturation adjustment (custom white level)
- no_auto_scale (bool) – Whether to disable pixel value scaling
- no_auto_bright (bool) – whether to disable automatic increase of brightness
- auto_bright_thr (float) – ratio of clipped pixels when automatic brighness increase is used (see no_auto_bright). Default is 0.01 (1%).
- adjust_maximum_thr (float) – see libraw docs
- bright (float) – brightness scaling
- highlight_mode (rawpy.HighlightMode | int) – highlight mode
- exp_shift (float) – exposure shift in linear scale. Usable range from 0.25 (2-stop darken) to 8.0 (3-stop lighter).
- exp_preserve_highlights (float) – preserve highlights when lightening the image with exp_shift. From 0.0 to 1.0 (full preservation).
- gamma (tuple) – pair (power,slope), default is (2.222, 4.5) for rec. BT.709
- chromatic_aberration (tuple) – pair (red_scale, blue_scale), default is (1,1), corrects chromatic aberration by scaling the red and blue channels
- bad_pixels_path (str) – path to dcraw bad pixels file. Each bad pixel will be corrected using the mean of the neighbor pixels. See the rawpy.enhance module for alternative repair algorithms, e.g. using the median.

HPC Processing script - NEF to JPEG

In [ ]:
##Copy into nano and save as 'convert_nef_to_jpeg.py'

import rawpy  # pip install in environment in Anaconda Prompt before use (pip install rawpy imageio pillow)
import imageio
import os
import glob
from PIL import Image
import argparse
from multiprocessing import Pool
import io

def process_file(nef_file, output_folder):
    # Generate the output JPEG file path
    file_name = os.path.basename(nef_file)
    jpeg_file_path = os.path.join(output_folder, os.path.splitext(file_name)[0] + '.jpg')

    # Read and convert the NEF file into memory
    with rawpy.imread(nef_file) as raw:
        rgb = raw.postprocess(use_camera_wb=True)

    # Convert the raw image to an in-memory file object using BytesIO
    img_data = io.BytesIO()
    imageio.imwrite(img_data, rgb, format="jpeg", quality=100)

    # Rewind the in-memory file object
    img_data.seek(0)

    # Open the in-memory image with Pillow directly from the BytesIO object
    with Image.open(img_data) as img:
        # Resize the image to 8256 x 5504 pixels
        img = img.resize((8256, 5504), Image.Resampling.LANCZOS)
        # Set the DPI to 300 and save the JPEG in the output folder
        img.save(jpeg_file_path, 'JPEG', quality=100, dpi=(300, 300))

    print(f"Converted {nef_file} to {jpeg_file_path}")

def convert_nef_to_jpeg(input_folder, output_folder, num_workers):
    # Find all NEF files in the input folder
    nef_files = glob.glob(os.path.join(input_folder, '*.NEF'))

    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Create a pool of workers to parallelize the conversion
    with Pool(processes=num_workers) as pool:
        pool.starmap(process_file, [(nef_file, output_folder) for nef_file in nef_files])

if __name__ == "__main__":
    # Setup argument parser
    parser = argparse.ArgumentParser(description="Convert NEF files to JPEG format")
    parser.add_argument("input_folder", type=str, help="Path to the folder containing NEF files")
    parser.add_argument("output_folder", type=str, help="Path to the folder where JPEG files will be saved")

    # Get number of workers from SLURM environment variable or default to CPU count
    num_workers = int(os.environ.get('SLURM_CPUS_ON_NODE', os.cpu_count()))

    # Convert NEF files in parallel
    args = parser.parse_args()
    convert_nef_to_jpeg(args.input_folder, args.output_folder, num_workers)


HPC Processing script - ARW to JPEG

In [ ]:
##Copy into nano and save as 'convert_arw_to_jpeg.py'

import rawpy
import imageio
import os
import argparse
from multiprocessing import Pool
from PIL import Image
import io

def process_file(arw_file, output_folder):
    # Generate the output JPEG file path
    file_name = os.path.basename(arw_file)
    jpeg_file_path = os.path.join(output_folder, os.path.splitext(file_name)[0] + '.jpg')

    # Read and convert the ARW file into memory
    with rawpy.imread(arw_file) as raw:
        rgb = raw.postprocess(use_camera_wb=True)

    # Convert the raw image to an in-memory file object using BytesIO
    img_data = io.BytesIO()
    imageio.imwrite(img_data, rgb, format="jpeg", quality=100)

    # Rewind the in-memory file object
    img_data.seek(0)

    # Open the in-memory image with Pillow directly from the BytesIO object
    with Image.open(img_data) as img:
        # Ensure no black lines by trimming any black edges
        img = img.crop(img.getbbox())

        # Rotate the image if necessary to ensure landscape orientation
        width, height = img.size
        if height > width:
            img = img.rotate(90, expand=True)
        
        # Resize the image to the desired size without stretching or adding borders
        target_size = (6192, 4128)
        img.thumbnail(target_size, Image.Resampling.LANCZOS)

        # Ensure the image exactly matches the target size by resizing and cropping to fit
        img = img.resize(target_size, Image.Resampling.LANCZOS)

        # Save the final JPEG
        img.save(jpeg_file_path, 'JPEG', quality=100, dpi=(260, 260))

    print(f"Converted {arw_file} to {jpeg_file_path}")

def convert_arw_to_jpeg(input_folder, output_folder, num_workers):
    # Find all ARW files in the input folder
    arw_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.lower().endswith('.arw')]

    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Create a pool of workers to parallelize the conversion
    with Pool(processes=num_workers) as pool:
        pool.starmap(process_file, [(arw_file, output_folder) for arw_file in arw_files])

if __name__ == "__main__":
    # Setup argument parser
    parser = argparse.ArgumentParser(description="Convert ARW files to JPEG format")
    parser.add_argument("input_folder", type=str, help="Path to the folder containing ARW files")
    parser.add_argument("output_folder", type=str, help="Path to the folder where JPEG files will be saved")

    # Get number of workers from SLURM environment variable or default to CPU count
    num_workers = int(os.environ.get('SLURM_CPUS_ON_NODE', os.cpu_count()))

    # Convert ARW files in parallel
    args = parser.parse_args()
    convert_arw_to_jpeg(args.input_folder, args.output_folder, num_workers)



HPC Processing slurm - NEF to JPEG

In [ ]:
#!/bin/bash
#SBATCH --job-name=nef_to_jpeg
#SBATCH --output=nef_to_jpeg_%j.out
#SBATCH --error=nef_to_jpeg_%j.err
#SBATCH --exclusive
#SBATCH --partition=cpuq
#SBATCH --time=01:00:00

# Load the conda module
module load conda/anaconda3

# Activate your conda environment
conda activate nef_conversion  # or 'conda activate nef_conversion' depending on your cluster setup


# Define input and output directories
INPUT_FOLDER="/net/cluster1-prod-hpcnfs.aims.gov.au/3d-ltmp/EcoRRAP/data/202402/ONMO/FR2/S/P1" #enter the path to the folder containing NEF files
OUTPUT_FOLDER="/net/cluster1-prod-hpcnfs.aims.gov.au/3d-ltmp/pipeline/EcoRRAP_HPC/test_output/ONMO_FR2S_P1" #enter the path to the output folder

# Run the Python script
python convert_nef_to_jpeg.py "$INPUT_FOLDER" "$OUTPUT_FOLDER"

HPC Processing slurm - ARW to JPEG

In [ ]:
#!/bin/bash
#SBATCH --job-name=arw_to_jpeg
#SBATCH --output=arw_to_jpeg_%j.out
#SBATCH --error=arw_to_jpeg_%j.err
#SBATCH --exclusive
#SBATCH --partition=cpuq
#SBATCH --time=01:00:00

# Load the conda module
module load conda/anaconda3

# Activate your conda environment
conda activate nef_conversion  # or 'conda activate nef_conversion' depending on your cluster setup


# Define input and output directories
INPUT_FOLDER="/net/cluster1-prod-hpcnfs.aims.gov.au/3d-ltmp/EcoRRAP/data/202402/ONMO/FR2/S/P1" #enter the path to the folder containing ARW files
OUTPUT_FOLDER="/net/cluster1-prod-hpcnfs.aims.gov.au/3d-ltmp/pipeline/EcoRRAP_HPC/test_output/ONMO_FR2S_P1" #enter the path to the output folder

# Run the Python script
python convert_arw_to_jpeg.py "$INPUT_FOLDER" "$OUTPUT_FOLDER"

Local processing script - NEF to JPEG

In [1]:
import rawpy  # pip install in environment in Anaconda Prompt before use (pip install rawpy imageio pillow)
import imageio
import os
import glob
from PIL import Image

def convert_nef_to_jpeg(input_folder, output_folder):
    # Find all NEF files in the input folder
    nef_files = glob.glob(os.path.join(input_folder, '*.NEF'))
    
    for nef_file in nef_files:
        # Generate the output JPEG file path
        file_name = os.path.basename(nef_file)
        jpeg_file_path = os.path.join(output_folder, os.path.splitext(file_name)[0] + '.jpg')
        
        # Read and convert the NEF file
        with rawpy.imread(nef_file) as raw:
            rgb = raw.postprocess(use_camera_wb=True)
        
        # Save the converted file temporarily
        temp_file_path = os.path.join(output_folder, 'temp.jpg')
        imageio.imwrite(temp_file_path, rgb, quality=100)
        
        # Open the temporary file with Pillow
        with Image.open(temp_file_path) as img:
            # Resize the image to 8256 x 5504 pixels
            img = img.resize((8256, 5504), Image.Resampling.LANCZOS)
            # Set the DPI to 300
            img.save(jpeg_file_path, 'JPEG', quality=100, dpi=(300, 300))
        
        # Remove the temporary file
        os.remove(temp_file_path)
        
        print(f"Converted {nef_file} to {jpeg_file_path}")

# Ask for user input for the input and output folders
input_folder = input("Enter the path to the input folder:")
output_folder = input("Enter the path to the output folder:")

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Convert all NEF files in the input folder
convert_nef_to_jpeg(input_folder, output_folder)

print("Conversion completed.")

Converted \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_input\EC1_9084.NEF to \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_output\EC1_9084.jpg
Converted \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_input\EC1_9085.NEF to \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_output\EC1_9085.jpg
Converted \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_input\EC1_9086.NEF to \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_output\EC1_9086.jpg
Converted \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_input\EC1_9087.NEF to \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_output\EC1_9087.jpg
Converted \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_input\EC1_9088.NEF to \\pearl\3d-ltmp\pipeline\EcoRRAP_HPC\test_output\EC1_9088.jpg
Conversion completed.


Local processing script - ARW to JPEG

Includes functionality to:
- Ensure size remains same and crop borders
- Rotate photos if collected in portrait orientation
- These steps not required for Nikon-collected images but were experienced with Sony conversion

In [ ]:
import rawpy
import imageio
import os
from PIL import Image

def convert_arw_to_jpeg(input_folder, output_folder):
    # Find all ARW files in the input folder
    arw_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.arw')]
        
    for arw_file in arw_files:
        arw_file_path = os.path.join(input_folder, arw_file)
        jpeg_file_name = os.path.splitext(arw_file)[0] + '.jpg'
        jpeg_file_path = os.path.join(output_folder, jpeg_file_name)
        
        # Read the ARW file using rawpy
        with rawpy.imread(arw_file_path) as raw:
            # Postprocess the raw data with slight auto brightness and adjusted gamma
            rgb_image = raw.postprocess(use_camera_wb=True)

        # Save the temporary RGB image using imageio
        temp_file_path = os.path.join(output_folder, 'temp.jpg')
        imageio.imwrite(temp_file_path, rgb_image, quality=100)

        # Open the temporary image with Pillow for resizing
        with Image.open(temp_file_path) as img:
            # Ensure no black lines appear by trimming any black edges
            img = img.crop(img.getbbox())  # Crop the image to remove any unnecessary black areas

            # Rotate the image if it's in portrait orientation
            if img.height > img.width:
                img = img.rotate(90, expand=True)

            # Resize the image to the desired size without stretching or adding borders
            target_size = (6192, 4128)
            img.thumbnail(target_size, Image.Resampling.LANCZOS)
            
            # Ensure the image exactly matches the target size by resizing and cropping to fit
            img = img.resize(target_size, Image.Resampling.LANCZOS)

            # Save the final JPEG
            img.save(jpeg_file_path, 'JPEG', quality=100, dpi=(260, 260))

        # Remove the temporary file
        os.remove(temp_file_path)

        print(f"Converted {arw_file} to {jpeg_file_path}")

# Get input and output folder paths
input_folder = input("Enter the path to the folder containing ARW files: ")
output_folder = input("Enter the path to the folder where you want to save JPEG files: ")

# Convert ARW to JPEG
convert_arw_to_jpeg(input_folder, output_folder)

print("Conversion completed.")

